In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Komoran

# 자연어 처리 순서
1. 데이터로드
2. 데이터 튜닝
3. 데이터 분할
4. 토큰화
5. 백터화
6. 모델 학습
7. 평가

In [3]:
# 데이터 로드
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [4]:
# 필요 없는 컬럼을 제외 -> id 제외 -> id에 중복 값이 존재하지 않는다.
df.drop('id', axis=1, inplace=True)

In [5]:
# 결측치가 존재하는가?
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   document  149995 non-null  object
 1   label     150000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.3+ MB


In [6]:
# 결측치가 15000개의 데이터 중 5개 결측치가 관찰
# 5개니까 굉장히 적은 양 -> 제외
df.dropna(inplace=True)

In [7]:
# 리뷰 데이터중 중복된 문장이 존재하는가? -> 확인? -> value_counts
df['document'].value_counts()

document
굿                                      181
good                                    92
최고                                      85
쓰레기                                     79
별로                                      66
                                      ... 
1%라도 기대했던 내가 죄인입니다 죄인입니다....             1
아직도 이 드라마는 내인생의 최고!                      1
패션에 대한 열정! 안나 윈투어!                       1
키이라 나이틀리가 연기하고자 했던건 대체 정신장애일까 틱장애일까      1
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나        1
Name: count, Length: 146182, dtype: int64

In [8]:
# 중복으로 만들어져있는 리뷰 문자들을 제외 -> 과적합 방지
df.drop_duplicates('document', inplace=True)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146182 entries, 0 to 149999
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   document  146182 non-null  object
 1   label     146182 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.3+ MB


In [10]:
# 학습 데이터와 학습 데이터로 데이터를 분할
X = df['document'].values
Y = df['label'].values

# train test 셋으로 분할 (분류 데이터 -> label의 비율을 유지)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)

In [11]:
# X의 데이터가 문자임으로 토근화 작업
komoran = Komoran()
# 사용할 품사 선택
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']
# 사용하지 않을 단어를 선택
stop_word = ['하다', '되다']
# 글자 수 제한
len_word = 2
# 토큰화 함수 정의
def tokenize(text):
    # 결고를 리스트의 형태로 되돌려주기 위해 빈 리스트를 생성
    tokens = []
    for word, pos in komoran.pos(text):
        # 조건1 : 품사에 포함되어있다면
        # 조건2 : 금지어에 포함되어있지 않다면
        # 조건3 : 문자열 길이가 len_word보다 크거나 같다면
        if pos in allow_pos and word not in stop_word and len(word) >= len_word:
            # 3개의 조건을 모두 만족하는 단어를 tokens에 추가
            tokens.append(word)
    return tokens

In [ ]:
# 백터화 객체 생성 (단어의 중요도를 판단하는 백터화 class 로드)
# 백터화(자연어 데이터에서 사용하는 스케일링 비슷한 작업)
vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=  (1, 1),
    min_df = 3,     # 3회 이상 나온 단어들을 기준으로 중요도 판단
    lowercase=False
)

In [13]:
# train 데이터를 이용하여 fit_transform()
# test 데이터는 transform() --> 데이터의 누수 방지

In [14]:
vectorizer.fit_transform(X_train)
print(len(vectorizer.get_feature_names_out()))

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


13575


In [16]:
vectorizer.fit_transform(X_test)
print(len(vectorizer.get_feature_names_out()))

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


5946
